# Experimental raw-label baseline
Results classify unverified raw values, not verified vulnerabilities. Review data rights before uploading. Do not attach the test split.


In [ ]:
from pathlib import Path
import sys
import json
import subprocess

input_root = Path('/kaggle/input')
candidates = sorted(p.parent for p in input_root.rglob('train.csv')
                    if (p.parent / 'validation.csv').is_file()
                    and (p.parent / 'train_baseline.py').is_file())
if len(candidates) != 1:
    raise RuntimeError(f'Attach exactly one private dataset with train.csv, validation.csv, and train_baseline.py together; found {len(candidates)}.')
data_dir = candidates[0]
script = data_dir / 'train_baseline.py'
print('Input directory:', data_dir)
print('Baseline script:', script)
print('Python:', sys.version)


In [ ]:
try:
    import sklearn
except ImportError as exc:
    raise RuntimeError('scikit-learn is missing in this Kaggle environment; use an environment where it is already available.') from exc
print('scikit-learn:', sklearn.__version__)
print('EXPERIMENTAL: raw label meanings are unverified.')


In [ ]:
metrics_path = Path('/kaggle/working/metrics.json')
if metrics_path.exists():
    raise FileExistsError(f'{metrics_path} already exists. Download or move it before rerunning.')
command = [sys.executable, str(script), '--experimental-raw-labels',
           '--data-dir', str(data_dir), '--train-limit', '2000',
           '--max-features', '5000', '--max-chars', '10000',
           '--metrics-output', str(metrics_path)]
subprocess.run(command, check=True)


In [ ]:
with metrics_path.open(encoding='utf-8') as stream:
    metrics = json.load(stream)
print('EXPERIMENTAL raw-label validation metrics:')
print(json.dumps(metrics, indent=2))
print('Saved:', metrics_path)
